# Supplementary Figs. 31-38 - longitudinal phased mCA detection at earlier timepoints

The TETRIS-seq **longitudinal phased** mCA caller. Once an mCA has
been called in a participant's index sample by `Supplementary_Fig_31-38-mCA_calling_unphased.ipynb`, the
haplotype carrying it is phased from that sample and the same phasing applied to every available earlier timepoint
from the same participant. Summing the B-allele frequency deviation along a known haplotype is far more
sensitive than looking for an unphased BAF split, so mCAs can be traced back to timepoints where the
unphased caller sees nothing.

This notebook produces **the lower panels of Supplementary Figs. 31-38** - one per earlier timepoint,
showing BAF and phased deviation across the affected chromosome, with the mean phased deviation tested by
a one-sided t-test (alpha = 0.05) and converted to a cell fraction with a 95% confidence interval. Each
panel is displayed inline as well as saved.

The method's sensitivity and false-positive rate were characterised on simulated data in
`Supplementary_Fig_28-29-mCA_phased_caller_benchmarking_on_simulated_samples.ipynb`, which is also where the shared functions
below were developed; they are reproduced here so this notebook runs on its own.

## Data availability

The phased caller reads **one file per sample-timepoint**: the per-SNP B-allele frequencies. These are
**individual-level participant data** and are **not distributed with this code** - a per-SNP BAF profile is
a germline genotype fingerprint. They are **deposited under controlled access in the European
Genome-phenome Archive** and released to approved researchers via a Data Access Committee.

No log R ratio file is needed here: phasing tracks the BAF deviation alone.

| file | used for | location |
|---|---|---|
| `EGA_deposit_CNV_BAF_LRR/*_variant_calling_only_SNPs_annovar_annotated.txt` | per-SNP BAF, for the reference and target timepoints | EGA, controlled access |
| `mCAs_for_phasing.csv` | the mCAs selected for longitudinal tracing | `Data_files/mCA_calling/Real_data/` |
| `chromosome_ideogram_hg19.txt` | chromosome ideograms drawn under each panel | Data_files |

The resulting per-timepoint cell fractions, confidence intervals and p-values are written to
`mCA_phased_longitudinal_long.csv` and are published in **Supplementary Table 7**.


**Deposit layout.** The BAF and LRR files are deposited as one directory, one file per
sample-timepoint per type - and are read from `Data_files/mCA_calling/Real_data/EGA_deposit_CNV_BAF_LRR/`.
Some filenames carry the library UDI suffix and some do not, so each file is located by sample prefix
(`find_deposit_file`) rather than by a constructed library/sample path.

All 32 sample-timepoints needed by Supplementary Figs. 31-38 are present in the deposit, so every panel in
the series can be regenerated.

## Configuration and shared functions

Chromosome sizes and ideograms, the phased cell-fraction estimator, the phasing derivation, and the
panel-plotting function. All are shared with the benchmarking notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
try:
except ImportError:
    BrokenBarHCollection = None
import os, re, warnings
from scipy.stats import ttest_1samp, t as t_dist

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
# === COLOR DEFINITIONS ===
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33); c2 = (1.00, 0.23, 0.19)
c3 = (1.00, 0.58, 0.00); c4 = (1.00, 0.80, 0.00)
c5 = (0.30, 0.85, 0.39); c6 = (0.35, 0.78, 0.98)
c7 = (0.20, 0.67, 0.86); c8 = (0.00, 0.48, 1.00)
c9 = (0.35, 0.34, 0.84); c10 = (0.00, 0.31, 0.57)

orange1='#feedde'; orange2='#fdbe85'; orange3='#fd8d3c'; orange4='#e6550d'; orange5='#a63603'
blue1='#eff3ff'; blue2='#bdd7e7'; blue3='#6baed6'; blue4='#3182bd'; blue5='#08519c'
green1='#edf8e9'; green2='#bae4b3'; green3='#74c476'; green4='#31a354'; green5='#006d2c'
grey1='#f7f7f7'; grey2='#cccccc'; grey3='#969696'; grey4='#636363'; grey5='#252525'
purple1='#f2f0f7'; purple2='#cbc9e2'; purple3='#9e9ac8'; purple4='#756bb1'; purple5='#54278f'
red1='#fee5d9'; red2='#fcae91'; red3='#fb6a4a'; red4='#de2d26'; red5='#a50f15'

In [ ]:
ideogram_file = 'Data_files/chromosome_ideogram_hg19.txt'

chromosome_sizes = {
    'chr1': 249250621, 'chr2': 243199373, 'chr3': 198022430, 'chr4': 191154276,
    'chr5': 180915260, 'chr6': 171115067, 'chr7': 159138663, 'chr8': 146364022,
    'chr9': 141213431, 'chr10': 135534747, 'chr11': 135006516, 'chr12': 133851895,
    'chr13': 115169878, 'chr14': 107349540, 'chr15': 102531392, 'chr16': 90354753,
    'chr17': 81195210, 'chr18': 78077248, 'chr19': 59128983, 'chr20': 63025520,
    'chr21': 48129895, 'chr22': 51304566, 'chrX': 155270560
}


In [ ]:
def ideograms(ideogram_file, chromosome):
    
    color_lookup = {'gneg': (1., 1., 1.),
                    'gpos25': (.6, .6, .6),
                    'gpos50': (.4, .4, .4),
                    'gpos75': (.2, .2, .2),
                   'gpos100': (0., 0., 0.),
                      'acen': (.8, .4, .4),
                      'gvar': (.8, .8, .8),
                     'stalk': (.9, .9, .9)}
    
    ideogram = open(ideogram_file)
    ideogram.readline()
    xranges = []
    colors = []
    mid_points = []
    labels = []

    for line in ideogram:
        chrom, start, stop, label, stain = line.strip().split('\t')
        start = int(start)
        stop = int(stop)
        width = stop - start
        mid_point = start + (width/2)
        if chrom == chromosome:
            xranges.append((start, width))
            colors.append(color_lookup[stain])
            mid_points.append(mid_point)
            labels.append(label)
        
    return xranges, [0, 0.9], colors, mid_points, labels

def plot_chromosome(ideogram_file, chromosome, ax):

    xranges, yrange, colors, midpoints, labels = ideograms(ideogram_file, chromosome)

    ax.broken_barh(xranges, yrange, facecolors= colors, edgecolor = 'black')

    ax.set_xticks(midpoints)
    ax.set_xticklabels(labels, rotation = 90, fontsize = 9)
    ax.set_yticks([])
    ax.text(-0.013, 0.35, chromosome, transform=ax.transAxes, fontsize = 15, ha = 'right')
    ax.xaxis.set_tick_params(width=0.8, color = grey3, length = 6)

    ax.minorticks_off()

    ax.spines['left'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    return ax

def load_centromeres(filepath):
    """
    Parses the UCSC cytoband file to find the start/end of centromeres ('acen').
    Returns a dictionary: {'1': (start, end), '2': (start, end), ...}
    """
    
    df = pd.read_csv(filepath, sep="\t", comment='#', header=None, 
                     names=['chrom', 'start', 'end', 'name', 'type'])
    
    # Filter for centromeric regions ('acen')
    acen = df[df['type'] == 'acen'].copy()
    
    centromeres = {}
    for chrom, grp in acen.groupby('chrom'):
        # Centromeres usually span two bands (p-arm end, q-arm start)
        # We take the overall min start and max end.
        start = grp['start'].min()
        end = grp['end'].max()
        
        centromeres[chrom] = {'start': start, 'end': end}
        
    return centromeres

centromere_dict = load_centromeres('Data_files/chromosome_ideogram_hg19.txt')
print("Loaded centromeres for:", list(centromere_dict.keys()))

In [ ]:
def phased_dev_to_cf(pd_val, event):
    """Convert mean phased_dev to cell fraction estimate."""
    if pd_val <= 0:
        return 0.0
    if event in ('CN-LOH', 'CNLOH'):
        return float(np.clip(2 * pd_val, 0, 1))
    elif event == 'GAIN':
        return float(np.clip(4 * pd_val / (1 - 2 * pd_val) if pd_val < 0.5 else 1.0, 0, 1))
    elif event == 'LOSS':
        return float(np.clip(4 * pd_val / (1 + 2 * pd_val), 0, 1))
    else:
        return float(np.clip(2 * pd_val, 0, 1))



In [ ]:
def plot_known_region_smaller_plot(member_snps, chrom, region_start, region_end, event,
                              phase_lookup,
                              cf_result=None, sample_name='',
                              het_lo=0.02, het_hi=0.98,
                              chromosome_sizes=None, ideogram_file=None,
                              save_path=None):
    """
    Plot a chromosome with phased SNPs colored by haplotype in the known region.
    AI panel removed. Compact layout: BAF (2x height) + phased deviation (1x) + ideogram.
    """

    # ── Event-specific colors ──
    hap_colors = {
        'GAIN':   {'plus': red4,    'minus': red2},
        'LOSS':   {'plus': blue4,   'minus': blue2},
        'CN-LOH': {'plus': orange3, 'minus': orange2},
    }
    default_colors = {'plus': grey5, 'minus': grey2}
    colors     = hap_colors.get(event, default_colors)
    color_plus  = colors['plus']
    color_minus = colors['minus']

    event_shading = {'GAIN': red1,    'LOSS': blue1,   'CN-LOH': orange1}
    event_border  = {'GAIN': red3,    'LOSS': blue3,   'CN-LOH': orange3}
    shade_color  = event_shading.get(event, grey1)
    border_color = event_border.get(event, grey3)

    # ── Prepare data ──
    d = member_snps[member_snps['chromosome'].astype(str) == str(chrom)].copy()
    d = d.sort_values('position').reset_index(drop=True)
    if d.empty:
        print(f"  No data for {chrom}")
        return None

    plot_het_lo = cf_result.get('eff_het_lo', het_lo) if cf_result else het_lo
    plot_het_hi = cf_result.get('eff_het_hi', het_hi) if cf_result else het_hi
    d['is_het'] = (d['VAF'] >= plot_het_lo) & (d['VAF'] <= plot_het_hi)
    d['phased_dev'] = np.nan
    d['haplotype']  = np.nan

    for idx, row in d.iterrows():
        pos = int(row['position'])
        if pos in phase_lookup:
            if not (plot_het_lo <= row['VAF'] <= plot_het_hi):
                continue
            phase = phase_lookup[pos]
            d.at[idx, 'phased_dev'] = (2 * phase - 1) * (row['VAF'] - 0.5)
            d.at[idx, 'haplotype']  = phase

    homs           = d[~d['is_het']].copy()
    in_region      = (d['position'] >= region_start) & (d['position'] <= region_end)
    het_outside    = d['is_het'] & ~in_region
    phased_in_region       = in_region & d['phased_dev'].notna()
    hap_A                  = phased_in_region & (d['haplotype'] == 1.0)
    hap_B                  = phased_in_region & (d['haplotype'] == 0.0)
    unphased_het_in_region = in_region & d['phased_dev'].isna() & d['is_het']

    # ── Significance info ──
    is_significant = False
    cf_text = ''
    p_text  = ''
    if cf_result is not None:
        is_significant = cf_result.get('significant_raw', False)
        cf_est = cf_result.get('cf_estimate', 0)
        cf_lo  = cf_result.get('cf_lower_95', None)
        cf_hi  = cf_result.get('cf_upper_95', None)
        p_val  = cf_result.get('p_onesample', 1)
        cf_text = f'CF={cf_est:.3f} ({cf_lo:.3f}–{cf_hi:.3f})' if cf_lo is not None else f'CF={cf_est:.3f}'
        p_text  = f'p={p_val:.2e}'

    # ── LRR availability ──
    has_lrr = 'lrr_filt' in d.columns and d['lrr_filt'].notna().sum() > 10

    # ── Figure layout ──
    # Without LRR: [BAF(4), phased(2), ideogram(0.75)]  → fig_height ~2.7
    # With    LRR: [LRR(3), BAF(4), phased(2), ideogram(0.75)] → fig_height ~3.3
    if has_lrr:
        height_ratios = [3, 4, 2, 0.6]
        fig_height    = 2.8
        n_panels      = 4
    else:
        height_ratios = [4, 2, 0.6]
        fig_height    = 2.2
        n_panels      = 3

    fig, axes = plt.subplots(
        n_panels, 1, figsize=(14, fig_height), sharex=False,
        gridspec_kw={'height_ratios': height_ratios, 'hspace': 0.06},
        constrained_layout=True
    )

    panel_idx = 0

    # ═══ PANEL: LRR (optional) ═══
    if has_lrr:
        ax_lrr = axes[panel_idx]; panel_idx += 1
        ax_lrr.scatter(d['position'], d['lrr_filt'], s=4, color=grey3, alpha=0.3, zorder=1)
        if hap_A.any():
            ax_lrr.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'lrr_filt'],
                           s=8, color=color_plus, alpha=0.8, zorder=10)
        if hap_B.any():
            ax_lrr.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'lrr_filt'],
                           s=8, color=color_minus, alpha=0.8, zorder=10)
        if unphased_het_in_region.any():
            ax_lrr.scatter(d.loc[unphased_het_in_region, 'position'],
                           d.loc[unphased_het_in_region, 'lrr_filt'],
                           s=6, color=grey4, alpha=0.5, zorder=5)
        ax_lrr.axhline(0, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
        ax_lrr.set_ylabel('LRR', fontsize=8)
        ax_lrr.set_ylim(-1.5, 1.5)

    # ═══ PANEL: BAF ═══
    ax_baf = axes[panel_idx]; panel_idx += 1
    if not homs.empty:
        ax_baf.scatter(homs['position'], homs['VAF'], s=4, color='lightgray', alpha=0.6, zorder=1)
    if het_outside.any():
        ax_baf.scatter(d.loc[het_outside, 'position'], d.loc[het_outside, 'VAF'],
                       s=6, color=grey4, alpha=0.8, zorder=5)
    if hap_A.any():
        ax_baf.scatter(d.loc[hap_A, 'position'], d.loc[hap_A, 'VAF'],
                       s=10, color=color_plus, alpha=0.9, zorder=10)
    if hap_B.any():
        ax_baf.scatter(d.loc[hap_B, 'position'], d.loc[hap_B, 'VAF'],
                       s=10, color=color_minus, alpha=0.9, zorder=10)
    if unphased_het_in_region.any():
        ax_baf.scatter(d.loc[unphased_het_in_region, 'position'],
                       d.loc[unphased_het_in_region, 'VAF'],
                       s=8, color=grey4, alpha=0.5, zorder=5)
    ax_baf.axhline(0.5, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
    ax_baf.set_ylabel('BAF', fontsize=8)
    ax_baf.set_ylim(-0.05, 1.05)

    # ═══ PANEL: Phased deviation ═══
    ax_phased = axes[panel_idx]; panel_idx += 1
    phased_hets = d[d['phased_dev'].notna()].copy()
    if not phased_hets.empty:
        pos_dev = phased_hets[phased_hets['haplotype'] == 1.0]
        neg_dev = phased_hets[phased_hets['haplotype'] == 0.0]
        if not pos_dev.empty:
            ax_phased.scatter(pos_dev['position'], pos_dev['phased_dev'],
                              s=10, color=color_plus, alpha=0.9, zorder=10)
        if not neg_dev.empty:
            ax_phased.scatter(neg_dev['position'], neg_dev['phased_dev'],
                              s=10, color=color_minus, alpha=0.9, zorder=10)
    ax_phased.axhline(0, linestyle='--', color=grey2, linewidth=0.8, zorder=0)
    ax_phased.set_ylabel('Phased\ndev.', fontsize=8)

    if cf_result is not None and not phased_hets.empty:
        mean_dev   = cf_result.get('mean_phased_dev', 0)
        line_color = color_plus if is_significant else grey3
        line_style = '-' if is_significant else ':'
        pos_range  = d['position'].max() - d['position'].min()
        ax_phased.axhline(
            mean_dev, linestyle=line_style, color=line_color,
            linewidth=1.5, alpha=0.8, zorder=15,
            xmin=(region_start - d['position'].min()) / pos_range,
            xmax=(region_end   - d['position'].min()) / pos_range,
        )

    # ═══ PANEL: Ideogram ═══
    ax_ideo = axes[panel_idx]
    try:
        plot_chromosome(ideogram_file, chrom, ax_ideo)
    except Exception:
        ax_ideo.set_xlim(0, chromosome_sizes.get(chrom, d['position'].max()) if chromosome_sizes else d['position'].max())
        ax_ideo.set_yticks([])
        # ax_ideo.text(0.5, 0.5, chrom, transform=ax_ideo.transAxes,
        #              ha='center', va='center', fontsize=10)

    if chromosome_sizes and chrom in chromosome_sizes:
        chrom_size_mb = chromosome_sizes[chrom] / 1e6
        # ax_ideo.text(1.01, 0.5, f"{chrom_size_mb:.0f} Mb",
                    #  transform=ax_ideo.transAxes, ha='left', va='center',
                    #  fontsize=9, color='black')
        ax_ideo.tick_params(axis='x', which='both', bottom=False, labelbottom=False)

    # ═══ Region shading + dashed outline ═══
    data_axes = [ax for ax in axes[:-1]]
    for ax in data_axes:
        ax.axvspan(region_start, region_end, color=shade_color, alpha=0.25, zorder=0)
        ymin, ymax = ax.get_ylim()
        rect = mpatches.Rectangle(
            (region_start, ymin), region_end - region_start, ymax - ymin,
            linewidth=1.2, edgecolor=border_color, facecolor='none',
            linestyle='--', zorder=20
        )
        ax.add_patch(rect)

    # ═══ Spine styling ═══
    for ax in axes:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.2)
        ax.spines['bottom'].set_linewidth(1.2)
        ax.spines['left'].set_color(grey3)
        ax.spines['bottom'].set_color(grey3)
        ax.tick_params(labelsize=7)

    # ═══ X-limits ═══
    chrom_size = chromosome_sizes.get(chrom, int(d['position'].max() * 1.05)) \
                 if chromosome_sizes else int(d['position'].max() * 1.05)
    for ax in axes:
        ax.set_xlim(0, chrom_size)

    for ax in axes[:-1]:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    # axes[-1].tick_params(axis='x', bottom=True, labelbottom=True)

    # ═══ Title ═══
    sig_str = '✓ SIGNIFICANT' if is_significant else '✗ not significant'
    title = f'{sample_name}\n{chrom}: {event} {region_start/1e6:.1f}–{region_end/1e6:.1f} Mb'
    if cf_result is not None:
        n_phased = cf_result.get('n_hets_inside', 0)
        title += f'  |  {cf_text}  |  {p_text}  |  {sig_str}  |  n={n_phased} phased hets'

    # ═══ Legend ═══
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_plus,
               markersize=7, label=f'Haplotype A ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=color_minus,
               markersize=7, label=f'Haplotype B ({event})'),
        Line2D([0], [0], marker='o', color='w', markerfacecolor=grey4,
               markersize=5, label='Unphased'),
        mpatches.Patch(facecolor=shade_color, edgecolor=border_color,
                       linestyle='--', label='Known region'),
    ]
    fig.suptitle(title, fontsize=9, fontweight='bold', x=0.02, ha='left')
    fig.legend(handles=legend_elements, fontsize=7, ncol=4,
               loc='upper right', bbox_to_anchor=(0.99, 0.99),
               framealpha=0.9, edgecolor='lightgrey')

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

    return fig

In [ ]:
def derive_phasing_from_vaf(snp_data, chrom, start, end, event_type=None,
                            min_hets=3, min_median_ai=0.01,
                            vaf_baselines=None):
    """
    Derive haplotype phasing from a high-CF sample's VAF.

    At high CF, het SNPs are pushed clearly above or below their baseline:
      VAF > baseline -> ALT on amplified haplotype -> phase = 1
      VAF < baseline -> ALT on other haplotype -> phase = 0

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    event_type : str or None - 'CN-LOH', 'GAIN', 'LOSS'
    min_hets : int - minimum het SNPs required
    min_median_ai : float - minimum median AI to accept phasing
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.

    Returns
    -------
    phase_lookup : dict {position: phase} or None if phasing fails
    n_phased : int - number of phased SNPs
    median_ai : float - median allelic imbalance in reference
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]
    region_snps = chrom_snps[
        (chrom_snps['position'] >= start) & (chrom_snps['position'] <= end)
    ]

    # VAF filter depends on event type
    # For CN-LOH/LOSS at high CF, original hets are pushed to VAF ~0 or ~1
    # (they look homozygous). We still need to phase them.
    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        ref_hets = region_snps.copy()
    else:
        ref_hets = region_snps[
            (region_snps['VAF'] >= 0.01) & (region_snps['VAF'] <= 0.99)
        ].copy()

    # deduplicate positions keeping highest AI row
    ref_hets = ref_hets.copy()
    ref_hets['_ai'] = np.abs(ref_hets['VAF'].astype(float) - 0.5)
    ref_hets = (ref_hets.sort_values('_ai', ascending=False)
                        .drop_duplicates(subset='position', keep='first')
                        .drop(columns='_ai'))

    if len(ref_hets) < min_hets:
        return None, len(ref_hets), 0

    # Get baseline for each position
    positions = ref_hets['position'].astype(int).values
    vafs = ref_hets['VAF'].astype(float).values

    if vaf_baselines is not None:
        bl = np.array([vaf_baselines.get((chrom_str, int(pos)), 0.5) for pos in positions])
    else:
        bl = np.full(len(positions), 0.5)

    # Check signal strength (AI relative to baseline)
    ai_values = np.abs(vafs - bl)
    median_ai = float(np.median(ai_values))
    if event_type not in ('CN-LOH', 'CNLOH', 'LOSS') and median_ai < min_median_ai:
        return None, len(ref_hets), median_ai

    # Derive phase relative to baseline
    phases = (vafs > bl).astype(float)
    phase_lookup = dict(zip(positions.tolist(), phases.tolist()))

    return phase_lookup, len(ref_hets), median_ai

def estimate_cf_phased(snp_data, chrom, start, end, phase_lookup, event_type,
                       het_lo=0.2, het_hi=0.8, min_hets=3, p_threshold=0.05,
                       vaf_baselines=None, n_permutations=10000, perm_seed=42,
                       baseline_weight=1.0):
    """
    Estimate cell fraction using externally-derived phasing.

    Computes phased_dev = (2*phase - 1) * (VAF - baseline) for each het SNP,
    then tests whether mean phased_dev is significantly > 0 using both a
    one-sided t-test and a sign-flip permutation test (both must yield
    P < p_threshold for the event to be called significant).

    Parameters
    ----------
    snp_data : DataFrame with chromosome, position, VAF columns
    chrom : str
    start, end : int - region coordinates
    phase_lookup : dict {position: phase} from reference sample
    event_type : str ('CN-LOH', 'GAIN', 'LOSS')
    het_lo, het_hi : float - VAF range for het filter
    min_hets : int - minimum phased het SNPs required
    p_threshold : float - significance threshold (applied to both tests)
    vaf_baselines : dict {(chrom, position): median_vaf} or None
        Population-level baseline VAFs. If None, uses 0.5 for all positions.
    n_permutations : int - number of sign-flip permutations
    perm_seed : int - random seed for permutation test

    Returns
    -------
    dict with cf_estimate, p_onesided, p_permutation, significant, n_hets,
         mean_phased_dev, t_stat
    or None if insufficient data
    """
    chrom_str = str(chrom)
    chrom_snps = snp_data[snp_data['chromosome'].astype(str) == chrom_str]

    if event_type in ('CN-LOH', 'CNLOH', 'LOSS'):
        
        def compute_devs(lo, hi):
            p_hets = chrom_snps[
                (chrom_snps['VAF'] >= lo) &
                (chrom_snps['VAF'] <= hi)
            ].copy()
            p_hets = p_hets[
                (p_hets['position'] >= start) &
                (p_hets['position'] <= end)
            ]
            devs = []
            for pos, vaf in zip(p_hets['position'].astype(int).values,
                                p_hets['VAF'].astype(float).values):
                if pos in phase_lookup:
                    phase = phase_lookup[pos]
                    bl = vaf_baselines.get((chrom_str, int(pos)), 0.5) if vaf_baselines else 0.5
                    bl_partial = 0.5 + baseline_weight * (bl - 0.5)
                    devs.append((2 * phase - 1) * (vaf - bl_partial))
            return devs
        
        region_size_mb = (end - start) / 1e6
        density_threshold = 0.5  # SNPs/Mb

        # Pass 1: 0.2-0.8
        devs = compute_devs(0.2, 0.8)
        pass_used = 1
        eff_het_lo_used, eff_het_hi_used = 0.2, 0.8
        density = len(devs) / region_size_mb

        # Pass 2: 0.05-0.95
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.05, 0.95)
            pass_used = 2
            eff_het_lo_used, eff_het_hi_used = 0.05, 0.95
            density = len(devs) / region_size_mb

        # Pass 3: 0.0-1.0
        if density < density_threshold or len(devs) < min_hets:
            devs = compute_devs(0.001, 0.999)
            pass_used = 3
            eff_het_lo_used, eff_het_hi_used = 0.001, 0.999

        # Set effective bounds to match whichever pass was used
        # (devs already computed — pass directly to phased_dev array)
        phased_dev = np.array(devs)
        n_hets = len(phased_dev)

        if n_hets < min_hets:
            return None

        # Skip the main het filter block below — already computed
        mean_dev = np.mean(phased_dev)
        t_stat, p_two = ttest_1samp(phased_dev, 0)
        p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2
        rng = np.random.default_rng(perm_seed)
        abs_vals = np.abs(phased_dev)
        null_means = np.array([
            np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
            for _ in range(n_permutations)
        ])
        p_permutation = float(np.mean(null_means >= mean_dev))
        significant = (p_onesided < p_threshold)
        cf_estimate = phased_dev_to_cf(mean_dev, event_type)

        se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
        t_crit = t_dist.ppf(0.975, df=n_hets - 1)
        cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
        cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)

        return {
            'cf_estimate': cf_estimate,
            'cf_lower_95': cf_lower_95,
            'cf_upper_95': cf_upper_95,
            'p_twosided': p_two,
            'p_onesided': p_onesided,
            'p_permutation': p_permutation,
            'significant': significant,
            'n_hets': n_hets,
            'mean_phased_dev': mean_dev,
            't_stat': t_stat,
            'pass_used': pass_used,
            'eff_het_lo': eff_het_lo_used,
            'eff_het_hi': eff_het_hi_used,
        }

    else:
        # GAIN — standard het filter, no multi-pass needed
        pass_used = 1
        eff_het_lo = het_lo
        eff_het_hi = het_hi
        eff_het_lo_used = het_lo
        eff_het_hi_used = het_hi

    hets = chrom_snps[
        (chrom_snps['VAF'] >= eff_het_lo) & (chrom_snps['VAF'] <= eff_het_hi)
    ].copy()

    if len(hets) == 0:
        return None

    # Filter to region + phased positions
    region_hets = hets[
        (hets['position'] >= start) & (hets['position'] <= end)
    ].copy()

    positions = region_hets['position'].astype(int).values
    vafs = region_hets['VAF'].astype(float).values

    phased_dev = []
    for pos, vaf in zip(positions, vafs):
        if pos in phase_lookup:
            phase = phase_lookup[pos]
            if vaf_baselines is not None:
                bl = vaf_baselines.get((chrom_str, int(pos)), 0.5)
            else:
                bl = 0.5
            bl_partial = 0.5 + baseline_weight * (bl-0.5)
            dev = (2 * phase - 1) * (vaf - bl_partial)
            phased_dev.append(dev)

    phased_dev = np.array(phased_dev)
    n_hets = len(phased_dev)

    if n_hets < min_hets:
        return None

    # --- One-sample t-test: is mean phased_dev > 0? ---
    mean_dev = np.mean(phased_dev)
    t_stat, p_two = ttest_1samp(phased_dev, 0)

    # One-sided p-value (we expect positive deviation)
    p_onesided = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    # --- Sign-flip permutation test ---
    rng = np.random.default_rng(perm_seed)
    abs_vals = np.abs(phased_dev)
    null_means = np.array([
        np.mean(rng.choice([-1, 1], size=n_hets) * abs_vals)
        for _ in range(n_permutations)
    ])
    p_permutation = float(np.mean(null_means >= mean_dev))

    # --- Significance requires just t-test (as testing in known regions) ---
    # significant = (p_onesided < p_threshold) and (p_permutation < p_threshold)
    significant = (p_onesided < p_threshold)

    # CF estimation
    cf_estimate = phased_dev_to_cf(mean_dev, event_type)
    se_dev = float(np.std(phased_dev, ddof=1)) / np.sqrt(n_hets)
    t_crit = t_dist.ppf(0.975, df=n_hets - 1)
    cf_lower_95 = phased_dev_to_cf(mean_dev - t_crit * se_dev, event_type)
    cf_upper_95 = phased_dev_to_cf(mean_dev + t_crit * se_dev, event_type)
    return {
        'cf_estimate': cf_estimate,
        'cf_lower_95': cf_lower_95,
        'cf_upper_95': cf_upper_95,
        'p_twosided': p_two,
        'p_onesided': p_onesided,
        'p_permutation': p_permutation,
        'significant': significant,
        'n_hets': n_hets,
        'mean_phased_dev': mean_dev,
        't_stat': t_stat,
        'pass_used': pass_used,      # 1, 2, or 3
        'eff_het_lo': eff_het_lo_used,
        'eff_het_hi': eff_het_hi_used,
    }

---
# Real Samples — Longitudinal Phased mCA Caller
For each mCA listed in `mCAs_for_phasing.csv`:
1. Derive phasing from the reference sample (the timepoint listed in the CSV)
2. Find all other timepoints for that patient across all library folders
3. Apply phased caller to every other timepoint
4. Plot and collate results

In [ ]:
MCAS_CSV = 'Data_files/mCA_calling/Real_data/mCAs_for_phasing.csv'

# The CNV BAF and LRR files as deposited on EGA: one flat directory, one file per
# sample-timepoint per type. Some filenames carry the library UDI suffix and some do not,
# so files are located by sample prefix rather than by a constructed path.
CNV_DEPOSIT_DIR = 'Data_files/mCA_calling/Real_data/EGA_deposit_CNV_BAF_LRR'
SNP_SUFFIX      = 'variant_calling_only_SNPs_annovar_annotated.txt'

# Output directory for plots
PLOT_DIR = 'Data_files/mCA_calling/Real_data/Longitudinal_phased_mCA_calls/PDFs'
os.makedirs(PLOT_DIR, exist_ok=True)

# Phasing parameters
# HET_LO        = 0.02
# HET_HI        = 0.98
HET_LO        = 0.2
HET_HI        = 0.8
MIN_HETS      = 3
MIN_MEDIAN_AI = 0.01
P_THRESHOLD   = 0.05


In [ ]:
# File finding functions

def parse_sample(sample_name):
    """
    Extract (patient_id, timepoint) from a sample name.
    Handles suffixes like 'C92_012_s7_CNV_xGenUDI1' -> ('C92_012', 7).
    """
    m = re.match(r'^((?:C92|CNTRL)_\d+)_[sS](\d+)', sample_name.strip())
    if m:
        return m.group(1), int(m.group(2))
    return None, None

def sample_base(sample_name):
    """'C92_012_s7_CNV_xGenUDI1' -> 'C92_012_s7'. The deposit is keyed on this."""
    m = re.match(r'^((?:C92|CNTRL)_\d+_[sS]\d+)', sample_name.strip())
    return m.group(1) if m else sample_name.split('_CNV')[0]



def load_snp_file(path):
    """Load a SNP file, normalising column names for the phased caller (needs 'VAF')."""
    df = pd.read_csv(path, sep='\t')
    # Unphased caller saves as BAF; phased caller needs VAF
    if 'VAF' not in df.columns and 'BAF' in df.columns:
        df = df.rename(columns={'BAF': 'VAF'})
    df['chromosome'] = df['chromosome'].astype(str)
    df['position']   = pd.to_numeric(df['position'], errors='coerce')
    df['VAF']        = pd.to_numeric(df['VAF'],      errors='coerce')

    # One row per genomic position - keep highest AI row
    df['_ai'] = np.abs(df['VAF'] - 0.5)
    df = (df.sort_values('_ai', ascending=False)
            .drop_duplicates(subset=['chromosome', 'position'], keep='first')
            .drop(columns='_ai')
            .reset_index(drop=True))
    return df

def find_all_timepoints(patient_id):
    """
    Every deposited timepoint for patient_id, from the flat deposit directory.
    Returns list of dicts: {timepoint, folder, snp_path}, ordered by timepoint.
    """
    found = {}
    for f in os.listdir(CNV_DEPOSIT_DIR):
        if not f.endswith(SNP_SUFFIX):
            continue
        pid, tp = parse_sample(f)
        if pid != patient_id or tp is None:
            continue
        found[tp] = {
            'timepoint': tp,
            'folder':    sample_base(f),
            'snp_path':  os.path.join(CNV_DEPOSIT_DIR, f),
        }
    return sorted(found.values(), key=lambda x: x['timepoint'])


### Run phased longitudinal pipeline on real samples

In [ ]:
mcas = pd.read_csv(MCAS_CSV)

all_results = []

for _, mca_row in mcas.iterrows():
    ref_sample_raw = str(mca_row['sample'])   # e.g. 'C92_022_s7' or 'C92_012_s7_CNV_xGenUDI1'
    chrom          = str(mca_row['chromosome'])
    start          = int(mca_row['start_pos'])
    end            = int(mca_row['end_pos'])
    event          = str(mca_row['event'])

    patient_id, ref_tp = parse_sample(ref_sample_raw)
    if patient_id is None:
        print(f"⚠️  Could not parse sample name: {ref_sample_raw}")
        continue

    print(f"\n{'='*65}")
    print(f"Patient: {patient_id}  |  {chrom} {event} {start/1e6:.1f}–{end/1e6:.1f} Mb  |  ref timepoint: s{ref_tp}")

    # ── 1. Find all timepoints for this patient ──────────────────────────
    timepoints = find_all_timepoints(patient_id)
    if not timepoints:
        print(f"  ⚠️  No timepoint files found for {patient_id}")
        continue

    tps_found = [f"s{t['timepoint']}" for t in timepoints]
    print(f"  Timepoints found: {tps_found}")

    # ── 2. Load reference sample SNP file ───────────────────────────────
    ref_entry = next((t for t in timepoints if t['timepoint'] == ref_tp), None)
    if ref_entry is None:
        print(f"  ⚠️  Reference timepoint s{ref_tp} not found on disk for {patient_id}")
        continue

    ref_snps = load_snp_file(ref_entry['snp_path'])
    print(f"  Reference: {ref_entry['folder']} ({len(ref_snps)} SNPs)")

    # ── 3. Derive phasing from reference sample ──────────────────────────
    phase_lookup, n_phased, median_ai = derive_phasing_from_vaf(
        ref_snps, chrom, start, end, event_type=event,
        min_hets=MIN_HETS, min_median_ai=MIN_MEDIAN_AI,
        vaf_baselines=None  # no population baseline for real samples (yet)
    )

    if phase_lookup is None:
        print(f"  ⚠️  Phasing failed for reference s{ref_tp} "
              f"(n_phased={n_phased}, median_ai={median_ai:.3f})")
        all_results.append({
            'patient': patient_id, 'chromosome': chrom,
            'start_pos': start, 'end_pos': end, 'event': event,
            'ref_timepoint': f"s{ref_tp}",
            'timepoint': f"s{ref_tp}",
            'role': 'reference',
            'phase_derived': False, 'n_phased': n_phased, 'median_ai_ref': median_ai,
            'n_hets': None, 'cf_estimate': None, 'p_onesided': None, 'significant': None,
        })
        continue

    print(f"  ✅ Phasing derived: {n_phased} SNPs, median AI={median_ai:.3f}")

    # # ── 4. Determine favored haplotype (majority phase = 1) ──────────────
    # phase_vals = list(phase_lookup.values())
    # favored_haplotype = 1 if sum(phase_vals) >= len(phase_vals) / 2 else 0

    # ── 5. Apply phased caller to ALL timepoints (including reference) ───
    for tp_entry in timepoints:
        tp        = tp_entry['timepoint']
        folder    = tp_entry['folder']
        role      = 'reference' if tp == ref_tp else 'earlier' if tp < ref_tp else 'later'

        snps = load_snp_file(tp_entry['snp_path'])

        cf_result = estimate_cf_phased(
            snps, chrom, start, end, phase_lookup, event,
            het_lo=HET_LO, het_hi=HET_HI, min_hets=MIN_HETS,
            p_threshold=P_THRESHOLD, vaf_baselines=None
        )

        if cf_result is None:
            print(f"  s{tp} ({role}): insufficient phased hets")
            cf_est = p_val = t_st = n_h = cf_lo = cf_hi = None
            sig = False
        else:
            cf_est = cf_result['cf_estimate']
            cf_lo  = cf_result['cf_lower_95']
            cf_hi  = cf_result['cf_upper_95']
            p_val  = cf_result['p_onesided']
            t_st   = cf_result['t_stat']
            n_h    = cf_result['n_hets']
            sig    = cf_result['significant']
            print(f"  s{tp} ({role}): CF={cf_est:.3f}, p={p_val:.4f}, "
                  f"n_hets={n_h}, {'✅ SIG' if sig else '— ns'}")

        all_results.append({
            'patient':       patient_id,
            'chromosome':    chrom,
            'start_pos':     start,
            'end_pos':       end,
            'event':         event,
            'ref_timepoint': f"s{ref_tp}",
            'timepoint':     f"s{tp}",
            'role':          role,
            'phase_derived': True,
            'n_phased':      n_phased,
            'median_ai_ref': median_ai,
            'n_hets':        n_h,
            'cf_estimate':   cf_est,
            'cf_lower_95':   cf_lo,
            'cf_upper_95':   cf_hi,
            'p_onesided':    p_val,
            't_stat':        t_st,
            'significant':   sig,
        })

        # Remap keys from estimate_cf_phased to what plot_known_region_phased expects
        if cf_result is not None:
            plot_cf_result = cf_result.copy()
            p_display = cf_result['p_onesided']
            plot_cf_result['p_onesample']   = p_display
            plot_cf_result['p_value']       = p_display
            plot_cf_result['p_permutation'] = p_display
            plot_cf_result['n_hets_inside']    = cf_result['n_hets']
            plot_cf_result['is_significant']   = cf_result['significant']
            plot_cf_result['significant_raw']  = cf_result['significant']  # ← this was missing
        else:
            plot_cf_result = None

        # ── 6. Plot ─────────────────────────────────────────────────────
        plot_fname = (f"{PLOT_DIR}/{patient_id}_{chrom}_{event}"
                      f"_ref_s{ref_tp}_target_s{tp}.pdf")
        try:
            fig = plot_known_region_smaller_plot(
                snps, chrom, start, end, event,
                phase_lookup,
                cf_result=plot_cf_result,
                sample_name=f"{folder}  ({role})",
                het_lo=HET_LO, het_hi=HET_HI,
                chromosome_sizes=chromosome_sizes,
                ideogram_file=ideogram_file,
                save_path=plot_fname
            )
            if fig is not None:
                #show the panel inline as well as saving it, so the figures are readable
                #in the notebook without re-running the caller
                display(fig)
                plt.close(fig)
        except Exception as e:
            print(f"    ⚠️  Plot failed for s{tp}: {e}")

results_df = pd.DataFrame(all_results)
print(f"\n{'='*65}")
print(f"Done. {len(results_df)} result rows across {results_df['patient'].nunique()} patients.")
display(results_df)


### Summary table and save plots

In [ ]:
# Wide-format: one row per patient/mCA, columns = timepoints
wide_rows = []
for (patient, chrom, event, ref_tp), grp in results_df.groupby(
        ['patient', 'chromosome', 'event', 'ref_timepoint']):

    row = {'patient': patient, 'chromosome': chrom,
           'event': event, 'ref_timepoint': ref_tp,
           'n_phased_snps': grp['n_phased'].iloc[0],
           'median_ai_ref': grp['median_ai_ref'].iloc[0]}

    for _, r in grp.sort_values('timepoint').iterrows():
        tp_label = r['timepoint']
        if r['role'] == 'reference':
            tp_label += ' (REF)'
        if r['cf_estimate'] is not None:
            tag = ' *' if r['significant'] else ''
            row[tp_label] = f"CF={r['cf_estimate']:.2f}{tag}"
        else:
            row[tp_label] = 'no data'

    wide_rows.append(row)

wide_df = pd.DataFrame(wide_rows)

# Sort timepoint columns numerically
def _tp_sort(col):
    m = re.search(r's(\d+)', col)
    return int(m.group(1)) if m else 999
fixed_cols = ['patient', 'chromosome', 'event', 'ref_timepoint',
              'n_phased_snps', 'median_ai_ref']
tp_cols = sorted([c for c in wide_df.columns if c not in fixed_cols], key=_tp_sort)
wide_df = wide_df[fixed_cols + tp_cols]

print("Legend: CF=X.XX = cell fraction estimate  |  * = significant (p<0.05)\n")
display(wide_df)

# Save both formats
os.makedirs('Data_files/mCA_calling/Real_data/Longitudinal_phased_mCA_calls', exist_ok=True)   #create the output folder on first run
results_df.to_csv('Data_files/mCA_calling/Real_data/Longitudinal_phased_mCA_calls/mCA_phased_longitudinal_long.csv', index=False)
wide_df.to_csv('Data_files/mCA_calling/Real_data/Longitudinal_phased_mCA_calls/mCA_phased_longitudinal_wide.csv', index=False)
print("\nSaved: mCA_phased_longitudinal_long.csv")
print("Saved: mCA_phased_longitudinal_wide.csv")
print(f"Plots: {PLOT_DIR}/")
